# 介紹

本程式使用 pyautogui 來完成 Biltz 的遊戲機器人

In [3]:
import pyautogui as pyg
pyg.sleep(2)
basex=1462
basey=310
gap=44


In [4]:
def get_gem_position(r, c):
    """回傳第 r 列、第 c 欄寶石中心的 (x, y) 座標（列、欄從 0 開始）。"""
    return basex + c * gap, basey + r * gap

In [5]:
def get_gem_color(r, c, im):
    """回傳截圖 im 中第 r 列、第 c 欄寶石中心的顏色。"""
    x, y = get_gem_position(r, c)
    return im.getpixel((x, y))

In [6]:
print(get_gem_position(0,0))
print(get_gem_position(0,1))
print(get_gem_position(1,0))


(1462, 310)
(1506, 310)
(1462, 354)


In [7]:
im=pyg.screenshot()
for i in range(8):
    x,y=get_gem_position(0,i)
    pyg.click(x,y,duration=1)
    print(get_gem_position(0,i))


(1462, 310)
(1506, 310)
(1550, 310)
(1594, 310)
(1638, 310)
(1682, 310)
(1726, 310)
(1770, 310)


In [8]:
im = pyg.screenshot()
for i in range(8):
    x, y = get_gem_position(0, i)
    pyg.click(x, y, duration=1)
    print(get_gem_color(0, i, im))

(25, 26, 27)
(25, 26, 27)
(25, 26, 27)
(25, 26, 27)
(25, 26, 27)
(25, 26, 27)
(25, 26, 27)
(25, 26, 27)


In [ ]:
for row in range(8):
    for col in range(8):
        x = basex + col * gap
        y = basey + row * gap
        color = pyg.pixel(x, y)
        print(row, col, x, y, color)

In [13]:
def can_move_left(r, c, im):
    """判斷第 r 列、第 c 欄的寶石左移一格後是否會形成三連消除。"""
    if not (0 <= r < 8 and 0 < c < 8):
        return False

    left_color = get_gem_color(r, c - 1, im)
    right_color = get_gem_color(r, c, im)

    def color_after_swap(row, col):
        if (row, col) == (r, c - 1):
            return right_color
        if (row, col) == (r, c):
            return left_color
        return get_gem_color(row, col, im)

    # 只需檢查包含交換位置的三格連線；其他位置不會因交換而改變。
    for row, col in ((r, c - 1), (r, c)):
        for dr, dc in ((0, 1), (1, 0)):
            for offset in (-2, -1, 0):
                cells = [(row + (offset + i) * dr, col + (offset + i) * dc)
                         for i in range(3)]
                if all(0 <= rr < 8 and 0 <= cc < 8 for rr, cc in cells):
                    colors = [color_after_swap(rr, cc) for rr, cc in cells]
                    if colors[0] == colors[1] == colors[2]:
                        return True
    return False

In [ ]:
def _can_swap_and_match(r, c, dr, dc, im):
    nr = r + dr
    nc = c + dc
    if not (0 <= r < 8 and 0 <= c < 8 and 0 <= nr < 8 and 0 <= nc < 8):
        return False

    current_color = get_gem_color(r, c, im)
    neighbor_color = get_gem_color(nr, nc, im)

    def color_after_swap(row, col):
        if (row, col) == (r, c):
            return neighbor_color
        if (row, col) == (nr, nc):
            return current_color
        return get_gem_color(row, col, im)

    def makes_match(row, col):
        color = color_after_swap(row, col)
        for axis_r, axis_c in ((0, 1), (1, 0)):
            count = 1
            for direction in (-1, 1):
                check_r = row + axis_r * direction
                check_c = col + axis_c * direction
                while (0 <= check_r < 8 and 0 <= check_c < 8
                       and color_after_swap(check_r, check_c) == color):
                    count += 1
                    check_r += axis_r * direction
                    check_c += axis_c * direction
            if count >= 3:
                return True
        return False

    return makes_match(r, c) or makes_match(nr, nc)

def can_move_right(r, c, im):
    return _can_swap_and_match(r, c, 0, 1, im)

def can_move_up(r, c, im):
    return _can_swap_and_match(r, c, -1, 0, im)

def can_move_down(r, c, im):

In [ ]:
import time

start_time = time.time()
im = pyg.screenshot()
directions = (
    (can_move_left, 0, -1),
    (can_move_right, 0, 1),
    (can_move_up, -1, 0),
    (can_move_down, 1, 0),
)

while time.time() - start_time < 60:
    moved = False
    for r in range(8):
        for c in range(8):
            for can_move, dr, dc in directions:
                if can_move(r, c, im):
                    x1, y1 = get_gem_position(r, c)
                    x2, y2 = get_gem_position(r + dr, c + dc)
                    pyg.moveTo(x1, y1)
                    pyg.dragTo(x2, y2, duration=0.15, button="left")
                    pyg.sleep(0.15)
                    im = pyg.screenshot()
                    moved = True
                    break
            if moved:
                break
        if moved:
            break
    if not moved:
        pyg.sleep(0.05)
        im = pyg.screenshot()

In [14]:
im=pyg.screenshot()
print(can_move_left(5,4,im))

False


In [ ]:
import pyautogui

# 輸入左上角寶石中心座標，以及相鄰寶石中心的間距
basex = int(input("左上角寶石中心的 x 座標 basex："))
basey = int(input("左上角寶石中心的 y 座標 basey："))
gap = int(input("相鄰寶石中心的間距 gap："))

# 擷取目前畫面，讀取 8×8 個寶石中心點的 RGB 顏色
screen = pyautogui.screenshot()

for row in range(8):
    colors = []
    for col in range(8):
        x = basex + col * gap
        y = basey + row * gap
        colors.append(screen.getpixel((x, y)))
    print(f"第 {row + 1} 行：{colors}")